In [1]:
# code/data/polygon_client.py
from polygon import RESTClient
from datetime import datetime
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()

class SnoogansPolygon:
    def __init__(self):
        api_key = os.getenv("POLYGON_API_KEY")
        if not api_key:
            raise ValueError("POLYGON_API_KEY not found in .env")
        self.client = RESTClient(api_key=api_key)
        print("Snoogans' eyes open — Polygon ready")

    def get_0dte_contracts(self, underlying="SPX", contract_type="put"):
        today = datetime.now().strftime("%Y-%m-%d")
        contracts = self.client.list_options_contracts(
            underlying_ticker=underlying,
            expiration_date=today,
            contract_type=contract_type,
            limit=1000
        )
        rows = []
        for c in contracts:
            rows.append({
                "ticker": c.ticker,
                "strike": c.strike_price,
                "type": c.contract_type,
                "bid": getattr(c.last_quote, "bid", None),
                "ask": getattr(c.last_quote, "ask", None),
                "delta": getattr(c, "delta", None),
                "iv": getattr(c, "implied_volatility", None)
            })
        df = pd.DataFrame(rows)
        if "delta" in df.columns:
            df["abs_delta"] = df["delta"].abs()
        print(f"Snoogans pulled {len(df)} {contract_type}s")
        return df

# Global instance
polygon = SnoogansPolygon()

Snoogans' eyes open — Polygon ready
